# The parasite inference atlases


**What this records (instructions 64.26, 64.27).** For Toxoplasma and Plasmodium, what each of the 39 catalogue methods has precomputed: held-out per-gene rows for genes with a known answer (each hidden with its orthogroup), deployment claims for genes without one, its calibration card -- or, where there is none, the specific reason and how it is run on demand. Then a check that the cache is real: a random sample of 4 cached (method, label) records per organism is rebuilt from scratch with the same frozen configuration and compared call by call.

Built 2026-11-07.

In [1]:
import os, json, shutil, numpy as np, pandas as pd
from parascape import atlas as AT, track_record as T, precompute_track_record as P
manifests = {org: AT.manifest(org) for org in ('Tg', 'Pf')}
for org, m in manifests.items():
    m.to_csv(os.path.join(OUT, f'manifest_{org}.tsv'), sep='\t', index=False)
pd.DataFrame({org: AT.coverage(m) for org, m in manifests.items()})

,Tg,Pf
methods,39,39
with_heldout_rows,14,14
with_deployment_claims,2,2
with_card,39,39
every_method_has_a_status,True,True


## 1 · Every method, per organism

Role, what is stored for known and for unknown genes, the card.

In [2]:
show = ['strategy', 'role', 'known_genes', 'unknown_genes', 'card']
manifests['Tg'][show]

,strategy,role,known_genes,unknown_genes,card
0,holdout_search,module,no per-gene label rows: a module method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured
1,geneset_hunt,gene set,no per-gene label rows: a gene set method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured
2,recoverability_atlas,gene ranking,no per-gene label rows: a gene ranking method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured
3,consensus_modules,module,no per-gene label rows: a module method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured
4,blind_battery,replication,no per-gene label rows: a replication method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured
5,block_ablation,gene label,not in the per-gene record for this organism (the method cannot speak on its labels),on demand: run from its module (results are cached by query identity),measured
6,feature_knn,gene label,"held-out rows for 8,140 genes on 9 labels","7,110 deployment claims on 1 labels",measured
7,map_neighbours,gene label,"held-out rows for 8,140 genes on 9 labels",on demand: deployment claims are built only for the best calibrated recipe of a label (claim_recipes.parquet); run it from Infer,measured
8,cluster_guilt,gene label,"held-out rows for 8,140 genes on 9 labels",on demand: deployment claims are built only for the best calibrated recipe of a label (claim_recipes.parquet); run it from Infer,measured
9,label_outliers,gene ranking,no per-gene label rows: a gene ranking method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured


In [3]:
manifests['Pf'][show]

,strategy,role,known_genes,unknown_genes,card
0,holdout_search,module,no per-gene label rows: a module method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured
1,geneset_hunt,gene set,no per-gene label rows: a gene set method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured
2,recoverability_atlas,gene ranking,no per-gene label rows: a gene ranking method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured
3,consensus_modules,module,no per-gene label rows: a module method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured
4,blind_battery,replication,no per-gene label rows: a replication method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured
5,block_ablation,gene label,not in the per-gene record for this organism (the method cannot speak on its labels),on demand: run from its module (results are cached by query identity),measured
6,feature_knn,gene label,"held-out rows for 5,720 genes on 3 labels",on demand: deployment claims are built only for the best calibrated recipe of a label (claim_recipes.parquet); run it from Infer,measured
7,map_neighbours,gene label,"held-out rows for 5,720 genes on 3 labels",620 deployment claims on 1 labels,measured
8,cluster_guilt,gene label,"held-out rows for 5,720 genes on 3 labels",on demand: deployment claims are built only for the best calibrated recipe of a label (claim_recipes.parquet); run it from Infer,measured
9,label_outliers,gene ranking,no per-gene label rows: a gene ranking method is graded by its own held-out scorecard (card),on demand: run from its module (results are cached by query identity),measured


## 2 · Canonical class queries already cached

The track record also holds every class of every label hidden as a whole set (mode `set`): the canonical 'what would it say about this class' query, stored once and read by query identity.

In [4]:
L = T.shipped()
sets = L[L['mode'].astype(str) == 'set']
canon = sets.groupby([sets.organism.astype(str), sets.target.astype(str)], observed=True).agg(
    methods=('strategy', 'nunique'), set_queries=('set_name', 'nunique'), rows=('gene_id', 'size'))
canon.to_csv(os.path.join(OUT, 'canonical_set_queries.tsv'), sep='\t')
canon

methods  set_queries    rows
organism target                                                    
Pf       is_exported                        14           22  110128
         lopit_pf_location                  14           38   57176
         pb_transferred_phenotype           14           23   64110
Tg       cellcycle_phase                    14           25   47278
         compartment                        14           44   83304
         dtm_class                          14           24  143966
         lopit_unified                      14           30   83500
         screen_actin_phenotype             14           18   11508
         screen_any_phenotype               14           18   11508
         screen_apicoplast_phenotype        14           18   11508
         screen_egress_phenotype            14           18   11508
         screen_replication_phenotype       14           18   11508

## 3 · Do cached records match a fresh run?

4 (method, label) pairs per organism, drawn at random (seed 7), rebuilt with `precompute_track_record.build_partition` (the production computation) and compared with the shipped rows: same genes, same call (an abstention on both sides counts as the same), same accuracy. The shipped record dates from 2026-10-17; any code change since that alters calls shows up here as a mismatch, and is reported, not hidden.

In [5]:
rng = np.random.default_rng(7)
together = L[L['mode'].astype(str) == 'together']
checks = []
work = os.path.join(OUT, '_fresh'); shutil.rmtree(work, ignore_errors=True)
for org in ('Tg', 'Pf'):
    pairs = together[together.organism.astype(str) == org][['strategy', 'target']].astype(str).drop_duplicates()
    pick = pairs.iloc[rng.choice(len(pairs), size=min(SAMPLES, len(pairs)), replace=False)]
    for key, target in pick.itertuples(index=False):
        part = P.plan(org, [target], [key])
        P.run(part, os.path.join(work, org))
        fresh = P.assemble(os.path.join(work, org), part)
        fresh = fresh[fresh['mode'].astype(str) == 'together'] if len(fresh) else fresh
        old = together[(together.organism.astype(str) == org) & (together.strategy.astype(str) == key) & (together.target.astype(str) == target)]
        old = old[old.setting_key.astype(str) == old.setting_key.astype(str).iloc[0]]
        j = old.merge(fresh, on='gene_id', suffixes=('_cached', '_fresh')) if len(fresh) else old.iloc[:0]
        same = (j.prediction_cached.astype(str) == j.prediction_fresh.astype(str)) | (j.prediction_cached.isna() & j.prediction_fresh.isna())
        checks.append(dict(organism=org, method=key, label=target, genes_cached=old.gene_id.nunique(),
                           genes_fresh=fresh.gene_id.nunique() if len(fresh) else 0, same_call=round(float(same.mean()), 4) if len(j) else float('nan'),
                           accuracy_cached=round(float(old.correct.fillna(False).mean()), 4),
                           accuracy_fresh=round(float(fresh.correct.fillna(False).mean()), 4) if len(fresh) else float('nan')))
checks = pd.DataFrame(checks); checks.to_csv(os.path.join(OUT, 'reproducibility.tsv'), sep='\t', index=False)
shutil.rmtree(work, ignore_errors=True)
checks

,organism,method,label,genes_cached,genes_fresh,same_call,accuracy_cached,accuracy_fresh
0,Tg,layer_vote,screen_replication_phenotype,318,318,1,0.3333,0.3333
1,Tg,supervised_classifier,screen_actin_phenotype,318,318,1,0.6792,0.6792
2,Tg,map_neighbours,screen_replication_phenotype,318,318,1,0.8616,0.8616
3,Tg,map_neighbours,screen_apicoplast_phenotype,318,318,1,0.805,0.805
4,Pf,cluster_guilt,lopit_pf_location,1580,1580,1,0.1437,0.1437
5,Pf,stratum_focus,lopit_pf_location,1580,1580,1,0.4918,0.4918
6,Pf,random_forest,lopit_pf_location,1580,1580,1,0.6456,0.6456
7,Pf,graph_convolution,lopit_pf_location,1580,1580,1,0.5937,0.5937
